# EEG Sleep Stage Preprocessing
- This cell sets up an EEG sleep analysis pipeline by importing required libraries.
- It defines directory paths for raw data, processed outputs, and figures.
- Creates a *Stage label map*. 


In [1]:
import os, glob, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mne
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, welch

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

# Paths
DATA_PATH   = "../data/raw/sleep-cassette"
OUTPUT_DIR  = "../data/processed"
FIGURES_DIR = "../outputs/figures"

os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Sleep stage label map 
STAGE_MAP = {
    "Sleep stage W": 0,  # Wake
    "Sleep stage 1": 1,  # NREM N1
    "Sleep stage 2": 2,  # NREM N2
    "Sleep stage 3": 3,  # NREM N3
    "Sleep stage 4": 3,  # NREM N4 → merged into N3 
    "Sleep stage R": 4,  # REM
}
LABEL_NAMES = {0:"Wake", 1:"N1", 2:"N2", 3:"N3", 4:"REM"}

# PSG and Hypnogram files 
psg_files = sorted(glob.glob(os.path.join(DATA_PATH, "**", "*PSG.edf"),       recursive=True))
hyp_files = sorted(glob.glob(os.path.join(DATA_PATH, "**", "*Hypnogram.edf"), recursive=True))

if not psg_files:
    psg_files = sorted(glob.glob(os.path.join(DATA_PATH, "*PSG.edf")))
    hyp_files = sorted(glob.glob(os.path.join(DATA_PATH, "*Hypnogram.edf")))

assert len(psg_files) > 0, f"No PSG files found in {DATA_PATH}"
assert len(psg_files) == len(hyp_files), "Mismatch between PSG and Hypnogram file counts"

print(f"Found {len(psg_files)} patient(s) in {DATA_PATH}")
for p, h in zip(psg_files, hyp_files):
    print(f"  {os.path.basename(p)}  +  {os.path.basename(h)}")

Found 153 patient(s) in ../data/raw/sleep-cassette
  SC4001E0-PSG.edf  +  SC4001EC-Hypnogram.edf
  SC4002E0-PSG.edf  +  SC4002EC-Hypnogram.edf
  SC4011E0-PSG.edf  +  SC4011EH-Hypnogram.edf
  SC4012E0-PSG.edf  +  SC4012EC-Hypnogram.edf
  SC4021E0-PSG.edf  +  SC4021EH-Hypnogram.edf
  SC4022E0-PSG.edf  +  SC4022EJ-Hypnogram.edf
  SC4031E0-PSG.edf  +  SC4031EC-Hypnogram.edf
  SC4032E0-PSG.edf  +  SC4032EP-Hypnogram.edf
  SC4041E0-PSG.edf  +  SC4041EC-Hypnogram.edf
  SC4042E0-PSG.edf  +  SC4042EC-Hypnogram.edf
  SC4051E0-PSG.edf  +  SC4051EC-Hypnogram.edf
  SC4052E0-PSG.edf  +  SC4052EC-Hypnogram.edf
  SC4061E0-PSG.edf  +  SC4061EC-Hypnogram.edf
  SC4062E0-PSG.edf  +  SC4062EC-Hypnogram.edf
  SC4071E0-PSG.edf  +  SC4071EC-Hypnogram.edf
  SC4072E0-PSG.edf  +  SC4072EH-Hypnogram.edf
  SC4081E0-PSG.edf  +  SC4081EC-Hypnogram.edf
  SC4082E0-PSG.edf  +  SC4082EP-Hypnogram.edf
  SC4091E0-PSG.edf  +  SC4091EC-Hypnogram.edf
  SC4092E0-PSG.edf  +  SC4092EC-Hypnogram.edf
  SC4101E0-PSG.edf  +  SC4101

### Inspect one file
- Chck a single EEG recording and its corresponding sleep stage annotation.
- Load the psg and hypnogram file.
- Extract metadata, and analyze the metadata by counting the sleep stage labels. 

In [2]:
raw_tmp = mne.io.read_raw_edf(psg_files[0], preload=False, verbose=False)
ann_tmp  = mne.read_annotations(hyp_files[0])
fs_check = raw_tmp.info["sfreq"]   # check sampling frequency (should be 100 Hz)

print(f"File     : {os.path.basename(psg_files[0])}")
print(f"Channels : {raw_tmp.ch_names}")
print(f"Sfreq    : {fs_check} Hz")
print(f"Duration : {raw_tmp.n_times / fs_check / 3600:.2f} h")
print()

# Count annotations by type
counts = {}
for a in ann_tmp:
    counts[a["description"]] = counts.get(a["description"], 0) + 1
print("Annotation types:")
for k, v in sorted(counts.items()):
    mapped = STAGE_MAP.get(k, "skipped")
    print(f"  {v:4d} x  '{k}'  →  label {mapped}")
del raw_tmp, ann_tmp

File     : SC4001E0-PSG.edf
Channels : ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']
Sfreq    : 100.0 Hz
Duration : 22.08 h

Annotation types:
    24 x  'Sleep stage 1'  →  label 1
    40 x  'Sleep stage 2'  →  label 2
    48 x  'Sleep stage 3'  →  label 3
    23 x  'Sleep stage 4'  →  label 3
     1 x  'Sleep stage ?'  →  label skipped
     6 x  'Sleep stage R'  →  label 4
    12 x  'Sleep stage W'  →  label 0


### Filtering
- **Bandpass 0.5–30 Hz** (Butterworth order 5, zero-phase) removes DC drift and high-frequency noise.  
- **Notch 50 Hz** is skipped automatically when `fs = 100 Hz` since the bandpass already eliminates it.

In [3]:
def bandpass(x, fs, lo=0.5, hi=30.0, order=5): 
    # Zero-phase Butterworth bandpass.
    sos = butter(order, [lo/(fs/2), hi/(fs/2)], btype="band", output="sos") # seond order 
    return sosfiltfilt(sos, x)

def notch(x, fs, f0=50.0, Q=30):
    # Auto-skip notch if f0 is above Nyquist frequency (fs/2) to avoid instability
    if f0 >= fs / 2:
        return x   # skip if notch frequency is above Nyquist
    b, a = iirnotch(f0 / (fs/2), Q)
    return filtfilt(b, a, x)

def filter_signal(x, fs):
    return notch(bandpass(x, fs), fs)

print("Filter functions ready.")
print(f"  Bandpass : 0.5–30 Hz, Butterworth order 5, zero-phase SOS")
print(f"  Notch    : 50 Hz Q=30 (auto-skipped when fs ≤ 100 Hz)")

Filter functions ready.
  Bandpass : 0.5–30 Hz, Butterworth order 5, zero-phase SOS
  Notch    : 50 Hz Q=30 (auto-skipped when fs ≤ 100 Hz)


### Visualize — raw vs filtered
- Saves `raw_vs_filtered.png`. Confirm the filter removes drift without distorting the EEG shape.

In [4]:
raw_vis = mne.io.read_raw_edf(psg_files[0], preload=True, verbose=False)
fs      = raw_vis.info["sfreq"]

# Pick the first EEG channel
eeg_chs = [ch for ch in raw_vis.ch_names if "EEG" in ch]
ch_name = eeg_chs[0] if eeg_chs else raw_vis.ch_names[0]

n   = int(30 * fs)                           # 30 seconds
x   = raw_vis.get_data(picks=[ch_name])[0, :n]
xf  = filter_signal(x, fs)
t   = np.arange(n) / fs

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 4), sharex=True)
ax1.plot(t, x  * 1e6, lw=0.5, color="#888"); ax1.set_title(f"Raw — {ch_name}"); ax1.set_ylabel("µV"); ax1.grid(alpha=0.3)
ax2.plot(t, xf * 1e6, lw=0.5, color="#1D9E75"); ax2.set_title("Filtered (0.5–30 Hz)"); ax2.set_ylabel("µV"); ax2.set_xlabel("s"); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "raw_vs_filtered.png"), dpi=150); plt.close()

print(f"Saved → {FIGURES_DIR}/raw_vs_filtered.png")
del raw_vis

Saved → ../outputs/figures/raw_vs_filtered.png


### Epoch + label + artifact rejection
- Slices the filtered signal into **30-second labeled windows**. 
- Rejects any epoch where peak-to-peak amplitude > 500 µV.

In [5]:
EPOCH_SEC    = 30      # clinical sleep-scoring standard
TARGET_FS    = 100.0  # resample to this if source differs
ART_THRESH   = 500e-6 # 500 µV in Volts — amplitude rejection threshold

def process_patient(psg_path, hyp_path):
    # Load one patient, filter, epoch, reject artifacts, return raw epochs.

    # Returns:
    # epochs : float32 (N, C, T)  — un-normalized, in Volts
    # labels : int64   (N,)
    # fs     : float

    raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
    fs  = raw.info["sfreq"]

    # Keep only EEG channels
    eeg_chs = [ch for ch in raw.ch_names if "EEG" in ch]
    if eeg_chs:
        raw.pick(eeg_chs)

    # Resample if needed
    if fs != TARGET_FS:
        raw.resample(TARGET_FS, verbose=False)
        fs = TARGET_FS

    # Filter every channel
    data = np.stack([filter_signal(raw.get_data()[i], fs)
                     for i in range(raw.get_data().shape[0])], axis=0)  # (C, T)

    spe  = int(EPOCH_SEC * fs)   # samples per epoch
    anns = mne.read_annotations(hyp_path)

    ep_list, lb_list, n_skip = [], [], 0
    for ann in anns:
        label = STAGE_MAP.get(ann["description"].strip())
        if label is None:
            n_skip += 1; continue

        s = int(ann["onset"] * fs)
        e = s + spe
        if e > data.shape[1] or (e - s) != spe:
            n_skip += 1; continue

        seg = data[:, s:e]

        # Amplitude artifact rejection
        ptp = seg.max(axis=1) - seg.min(axis=1)   # peak-to-peak per channel
        if ptp.max() > ART_THRESH:
            n_skip += 1; continue

        ep_list.append(seg)
        lb_list.append(label)

    epochs = np.array(ep_list, dtype=np.float32)
    labels = np.array(lb_list, dtype=np.int64)

    n_art = sum(1 for ann in anns
                if STAGE_MAP.get(ann["description"].strip()) is not None
                and int(ann["onset"]*fs)+spe <= data.shape[1]) - len(ep_list)

    print(f"  {os.path.basename(psg_path)}: {len(epochs)} epochs kept, {n_skip} skipped")
    return epochs, labels, fs

print("process_patient() ready.")

process_patient() ready.


### Z-score normalization
- Normalizes each epoch and channel independently: `(x − mean) / std`. 
- Removes amplitude differences between patients and nights.

In [6]:
def zscore(epochs):
    # Per-epoch per-channel z-score. Shape (N, C, T) → (N, C, T).
    mean = epochs.mean(axis=2, keepdims=True)
    std  = epochs.std( axis=2, keepdims=True)
    return (epochs - mean) / (std + 1e-8)

print("zscore() ready.")

zscore() ready.


### Run for all the patients and save

In [7]:
all_X, all_y = [], []
last_fs = None

for i, (psg, hyp) in enumerate(zip(psg_files, hyp_files)):
    epochs_raw, labels, fs = process_patient(psg, hyp)
    last_fs = fs

    epochs_norm = zscore(epochs_raw)

    # Basic sanity checks
    assert not np.isnan(epochs_norm).any(), f"NaN in patient {i+1}"
    assert not np.isinf(epochs_norm).any(), f"Inf in patient {i+1}"

    out = os.path.join(OUTPUT_DIR, f"patient_{i+1:02d}.npz")
    np.savez(out, X=epochs_norm, y=labels)

    all_X.append(epochs_norm)
    all_y.append(labels)

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)
np.savez(os.path.join(OUTPUT_DIR, "all_patients.npz"), X=X_all, y=y_all)

print(f"\nDone. all_patients.npz → X={X_all.shape}, y={y_all.shape}")
print(f"X mean={X_all.mean():.4f}  std={X_all.std():.4f}  NaN={np.isnan(X_all).sum()}")
print()
counts = np.bincount(y_all, minlength=5)
for idx, name in LABEL_NAMES.items():
    pct = 100 * counts[idx] / len(y_all)
    print(f"  {name:5s}: {counts[idx]:5d} epochs ({pct:4.1f}%)")

  SC4001E0-PSG.edf: 153 epochs kept, 1 skipped
  SC4002E0-PSG.edf: 150 epochs kept, 2 skipped
  SC4011E0-PSG.edf: 125 epochs kept, 1 skipped
  SC4012E0-PSG.edf: 170 epochs kept, 1 skipped
  SC4021E0-PSG.edf: 160 epochs kept, 1 skipped
  SC4022E0-PSG.edf: 177 epochs kept, 2 skipped
  SC4031E0-PSG.edf: 118 epochs kept, 1 skipped
  SC4032E0-PSG.edf: 122 epochs kept, 1 skipped
  SC4041E0-PSG.edf: 159 epochs kept, 2 skipped
  SC4042E0-PSG.edf: 173 epochs kept, 5 skipped
  SC4051E0-PSG.edf: 129 epochs kept, 1 skipped
  SC4052E0-PSG.edf: 136 epochs kept, 3 skipped
  SC4061E0-PSG.edf: 77 epochs kept, 1 skipped
  SC4062E0-PSG.edf: 97 epochs kept, 1 skipped
  SC4071E0-PSG.edf: 115 epochs kept, 1 skipped
  SC4072E0-PSG.edf: 178 epochs kept, 1 skipped
  SC4081E0-PSG.edf: 141 epochs kept, 1 skipped
  SC4082E0-PSG.edf: 155 epochs kept, 1 skipped
  SC4091E0-PSG.edf: 137 epochs kept, 11 skipped
  SC4092E0-PSG.edf: 103 epochs kept, 14 skipped
  SC4101E0-PSG.edf: 61 epochs kept, 2 skipped
  SC4102E0-PSG

### Plots

These plots help visualize key aspects of the data and model behavior.  
All figures are saved in `FIGURES_DIR`.
#### 1. Class Distribution
**File:** `class_distribution.png`  
Shows the number of epochs for each sleep stage.
#### 2. Epochs per Stage
**File:** `epochs_per_stage.png`  
Displays one representative waveform for each sleep stage.
#### 3. Mean Power Spectral Density (PSD)
**File:** `mean_psd_per_stage.png`  
Illustrates the average power spectrum for each stage.  
> Expectation: N3 stage should show a peak in the delta band (0.5–4 Hz).

In [8]:
COLORS = ["#3B8BD4","#1D9E75","#BA7517","#A32D2D","#534AB7"]
t_ax   = np.arange(X_all.shape[2]) / last_fs

# Plot 1: Class distribution 
counts = np.bincount(y_all, minlength=5)
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar([LABEL_NAMES[i] for i in range(5)], counts, color=COLORS, edgecolor="white")
for bar, c in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            f"{c}\n({100*c/len(y_all):.1f}%)", ha="center", va="bottom", fontsize=8)
ax.set_title("Class distribution"); ax.set_ylabel("Epochs"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,"class_distribution.png"), dpi=150); plt.close()

# Plot 2: One epoch per stage 
n_ch = X_all.shape[1]
fig, axes = plt.subplots(5, n_ch, figsize=(13, 10))
if n_ch == 1: axes = axes.reshape(5, 1)
for si in range(5):
    idxs = np.where(y_all == si)[0]
    for ci in range(n_ch):
        ax = axes[si, ci]
        if len(idxs): ax.plot(t_ax, X_all[idxs[0], ci], lw=0.5, color=COLORS[si])
        else:         ax.text(0.5,0.5,"no data",ha="center",va="center",transform=ax.transAxes,color="gray")
        ax.grid(alpha=0.2)
        if ci == 0:      ax.set_ylabel(LABEL_NAMES[si], fontweight="bold", fontsize=9)
        if si == 4:      ax.set_xlabel("s")
        if si == 0:      ax.set_title(f"Ch {ci}")
plt.suptitle("One epoch per stage (z-scored)", fontsize=11)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,"epochs_per_stage.png"), dpi=150); plt.close()

# Plot 3: Mean PSD per stage 
fig, axes = plt.subplots(1, n_ch, figsize=(13, 4))
if n_ch == 1: axes = [axes]
for ci in range(n_ch):
    ax = axes[ci]
    for si in range(5):
        idxs = np.where(y_all == si)[0]
        if not len(idxs): continue
        psds = [welch(X_all[ix, ci], fs=last_fs, nperseg=int(last_fs*4))[1]
                for ix in idxs[:40]]
        f_w  = welch(X_all[idxs[0], ci], fs=last_fs, nperseg=int(last_fs*4))[0]
        ax.semilogy(f_w, np.mean(psds,0), color=COLORS[si], label=LABEL_NAMES[si], lw=1.2)
    for lo_b,hi_b,nb_ in [(0.5,4,"δ"),(4,8,"θ"),(8,13,"α"),(13,30,"β")]:
        ax.axvspan(lo_b,hi_b,alpha=0.05,color="gray")
        ax.text((lo_b+hi_b)/2, ax.get_ylim()[1]*0.6, nb_, ha="center", fontsize=8, color="gray")
    ax.set_xlim(0,35); ax.set_xlabel("Hz"); ax.set_ylabel("PSD"); ax.set_title(f"Mean PSD — Ch {ci}")
    ax.legend(fontsize=8); ax.grid(alpha=0.2)
plt.suptitle("Mean PSD per stage  (N3 should peak in delta)", fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,"mean_psd_per_stage.png"), dpi=150); plt.close()

print(f"Plots saved to {FIGURES_DIR}/")
print("  class_distribution.png")
print("  epochs_per_stage.png")
print("  mean_psd_per_stage.png")
print()
print("Preprocessing complete. Next: model training with all_patients.npz")

Plots saved to ../outputs/figures/
  class_distribution.png
  epochs_per_stage.png
  mean_psd_per_stage.png

Preprocessing complete. Next: model training with all_patients.npz
